# 012-slic-superpixel-voting

# Imports

In [ ]:
import itertools
import os
from dataclasses import asdict

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.segmentation import find_boundaries

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.boundary import BOUNDARY_IOU_DILATION_PX
from uavloc.segmentation.eval import (MEAN_KEY, VAL_SETS, RefineFn, bilinear_argmax, evaluate, evaluate_predictions,
                                      load_baseline_model, load_val_sets, lraspp_outputs, run_model)
from uavloc.segmentation.latency import refiner_latency, standard_tile
from uavloc.segmentation.refine.superpixels import (SuperpixelMethod, VoteConfig, superpixel_refiner, superpixels,
                                                    vote)
from uavloc.track import experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "012-slic-superpixel-voting"
NOTE = ""
LOG_RUN = True

REFINER = "superpixels"            # "baseline" | "superpixels"
METHODS = ["slic", "felzenszwalb", "watershed"]
SLIC_CELLS_PX = [6, 10, 16]
SLIC_COMPACTNESS = [5, 10, 20]
FELZENSZWALB_SCALES = [25, 100]
FELZENSZWALB_CELL_PX = 6
WATERSHED_CELLS_PX = [6, 10, 16]
VOTES = ["all", "area"]
ALPHAS = [1.0, 0.5]

TUNE_SET = "birky-strip"
MIOU_GUARD = 0.01
BIOU_BAR = 0.010
LATENCY_BUDGET_MS = 114.0

SEED = 42
DEVICE = "cpu"
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_THREADS = 4
LATENCY_WARMUP = 3
LATENCY_RUNS = 15
ZOOMS = [["birky-strip-google", 0, [120, 1400, 376, 1656]], ["birky-strip", 3, [200, 2300, 456, 2556]],
         ["bushcha", 0, [1100, 1100, 1356, 1356]]]   # set, image index, [top, left, bottom, right] px

# Data

In [ ]:
set_seed(SEED)
artifacts = experiment_dir(EXP) / "artifacts"

In [ ]:
val_sets = load_val_sets()
{name: [a.rgb.shape[:2] for a in items] for name, items in val_sets.items()}

# Model

In [ ]:
model = load_baseline_model(DEVICE)

In [ ]:
cases = run_model(model, val_sets, DEVICE)

# Run

In [ ]:
baseline = evaluate(bilinear_argmax, cases)

In [ ]:
def grid_methods() -> list[SuperpixelMethod]:
    """Every registered over-segmentation whose method is in METHODS."""
    methods = ([SuperpixelMethod("slic", c, k) for c, k in itertools.product(SLIC_CELLS_PX, SLIC_COMPACTNESS)]
               + [SuperpixelMethod("felzenszwalb", FELZENSZWALB_CELL_PX, scale=s) for s in FELZENSZWALB_SCALES]
               + [SuperpixelMethod("watershed", c) for c in WATERSHED_CELLS_PX])
    return [m for m in methods if m.method in METHODS] if REFINER == "superpixels" else []

In [ ]:
methods = grid_methods()
votes = [VoteConfig(v, a) for v, a in itertools.product(VOTES, ALPHAS)]
tune_cases = {TUNE_SET: cases[TUNE_SET]}
segments = {m: [superpixels(c.image.rgb, m) for c in tune_cases[TUNE_SET]] for m in methods}
len(methods), len(votes)

In [ ]:
def scalar_row(scores: dict) -> dict:
    """Scalar metrics plus per-class IoU and Boundary IoU as flat columns."""
    row = {k: v for k, v in scores.items() if not isinstance(v, dict)}
    for metric in ("iou", "biou"):
        row.update({f"{metric}_{c}": v for c, v in scores[f"{metric}_per_class"].items()})
    return row

In [ ]:
def score_on_tune_set(method: SuperpixelMethod, config: VoteConfig) -> dict:
    """One grid cell scored on TUNE_SET, reusing the cached superpixels."""
    preds = [vote(c.output.probs, s, config) for c, s in zip(tune_cases[TUNE_SET], segments[method])]
    scores = evaluate_predictions({TUNE_SET: preds}, tune_cases)[TUNE_SET]
    return {**asdict(method), **asdict(config), **scalar_row(scores)}

In [ ]:
grid = pd.DataFrame([score_on_tune_set(m, v) for m, v in itertools.product(methods, votes)])
tune_baseline = scalar_row(baseline[TUNE_SET])
grid.sort_values("biou", ascending=False).round(4).head(20) if len(grid) else grid

In [ ]:
def select(grid: pd.DataFrame) -> tuple[SuperpixelMethod, VoteConfig] | None:
    """Highest TUNE_SET Boundary IoU among configs within MIOU_GUARD of the baseline's TUNE_SET mIoU."""
    eligible = grid[grid["miou"] >= tune_baseline["miou"] - MIOU_GUARD] if len(grid) else grid
    if eligible.empty:
        return None
    best = eligible.loc[eligible["biou"].idxmax()]
    return (SuperpixelMethod(best["method"], int(best["cell_px"]), float(best["compactness"]), float(best["scale"])),
            VoteConfig(best["vote"], float(best["alpha"])))

In [ ]:
chosen = select(grid)
refine_fn: RefineFn = superpixel_refiner(*chosen) if chosen else bilinear_argmax
results = evaluate(refine_fn, cases) if chosen else baseline
chosen

In [ ]:
def summary_table(runs: dict[str, dict]) -> pd.DataFrame:
    """One row per (refiner, val set); scalars plus per-class IoU / Boundary IoU."""
    rows = {(name, s): scalar_row(r[s]) if s != MEAN_KEY else r[s]
            for name, r in runs.items() for s in [*VAL_SETS, MEAN_KEY]}
    return pd.DataFrame(rows).T

In [ ]:
summary = summary_table({"baseline": baseline, "refined": results})
summary.round(3)

# Latency

In [ ]:
tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
tile_output = lraspp_outputs(model, tile, "cpu")
latency_kwargs = {"warmup": LATENCY_WARMUP, "runs": LATENCY_RUNS, "threads": LATENCY_THREADS}
argmax_latency = refiner_latency(bilinear_argmax, tile_output, tile, **latency_kwargs)
refine_latency = refiner_latency(refine_fn, tile_output, tile, **latency_kwargs)
argmax_latency.median_ms, refine_latency.median_ms

In [ ]:
def method_latency_ms(method: SuperpixelMethod) -> float:
    """Median CPU ms of the over-segmentation alone on the latency tile."""
    return refiner_latency(superpixel_refiner(method, VoteConfig()), tile_output, tile, **latency_kwargs).median_ms

In [ ]:
grid_latency = pd.DataFrame([{**asdict(m), "refine_ms": method_latency_ms(m)} for m in methods])
grid_latency.round(1)

# Metrics

In [ ]:
def run_metrics() -> dict[str, float]:
    """Mean and per-set scalars, per-class IoU/BIoU per set, latency, and the pre-registered checks."""
    metrics = {f"{k}_{s}".replace("-", "_"): v for s in [*VAL_SETS, MEAN_KEY]
               for k, v in (scalar_row(results[s]) if s != MEAN_KEY else results[s]).items()}
    metrics["boundary_iou_mean"] = results[MEAN_KEY]["biou"]
    metrics["boundary_iou_gain"] = results[MEAN_KEY]["biou"] - baseline[MEAN_KEY]["biou"]
    metrics["miou_change"] = results[MEAN_KEY]["miou"] - baseline[MEAN_KEY]["miou"]
    metrics["refine_ms_512"] = refine_latency.median_ms
    metrics["argmax_ms_512"] = argmax_latency.median_ms
    metrics["clears_bar"] = bool(metrics["boundary_iou_gain"] >= BIOU_BAR and metrics["miou_change"] >= -MIOU_GUARD
                                 and metrics["refine_ms_512"] <= LATENCY_BUDGET_MS)
    return metrics

In [ ]:
metrics = run_metrics()
{k: round(v, 4) for k, v in metrics.items() if k.endswith("_mean") or "ms" in k or k in ("boundary_iou_gain",
                                                                                          "miou_change", "clears_bar")}

# Plots

In [ ]:
def colour(ids: np.ndarray) -> np.ndarray:
    """Class ids to RGB with SEG_PALETTE; ignored pixels white."""
    out = SEG_PALETTE[np.minimum(ids, len(CLASSES) - 1)]
    out[ids == IGNORE_LABEL] = 255
    return out

In [ ]:
def zoom_panels(set_name: str, index: int, box: list[int]) -> list[np.ndarray]:
    """RGB with superpixel edges, label, baseline, refined for one crop."""
    case = cases[set_name][index]
    top, left, bottom, right = box
    rgb = case.image.rgb[top:bottom, left:right]
    probs = case.output.probs[:, top:bottom, left:right]
    edges_rgb = rgb.copy()
    if chosen:
        edges_rgb[find_boundaries(superpixels(rgb, chosen[0]))] = 255
    refined = refine_fn(case.output, case.image.rgb)[top:bottom, left:right]
    return [edges_rgb, colour(case.image.label[top:bottom, left:right]), colour(probs.argmax(0)), colour(refined)]

In [ ]:
titles = ["RGB + superpixels", "label", "baseline", "superpixel vote"]
fig_zoom, axes = plt.subplots(len(ZOOMS), len(titles), figsize=(4 * len(titles), 4 * len(ZOOMS)), squeeze=False)
for row, (set_name, index, box) in zip(axes, ZOOMS):
    for ax, panel, title in zip(row, zoom_panels(set_name, index, box), titles):
        ax.imshow(panel, interpolation="nearest"); ax.set_title(f"{title} ({set_name})", fontsize=9); ax.axis("off")
fig_zoom.suptitle(f"{chosen}" if chosen else "baseline", fontsize=8)
fig_zoom.tight_layout()

In [ ]:
fig_grid, ax = plt.subplots(figsize=(7, 5))
ax.axvline(tune_baseline["miou"], color="k", ls=":", lw=1); ax.axhline(tune_baseline["biou"], color="k", ls=":", lw=1)
ax.axvline(tune_baseline["miou"] - MIOU_GUARD, color="r", ls="--", lw=1)
for (method, vote_name), group in grid.groupby(["method", "vote"]) if len(grid) else []:
    ax.scatter(group["miou"], group["biou"], label=f"{method}, {vote_name}", marker="o" if vote_name == "all" else "^")
ax.set_xlabel(f"mIoU ({TUNE_SET})"); ax.set_ylabel(f"Boundary IoU ({TUNE_SET})"); ax.grid(alpha=.3)
ax.legend(fontsize=8) if len(grid) else None
fig_grid.tight_layout()

# Log

In [ ]:
RUN_CONFIG = {
    "refiner": REFINER, "methods": METHODS, "slic_cells_px": SLIC_CELLS_PX, "slic_compactness": SLIC_COMPACTNESS,
    "felzenszwalb_scales": FELZENSZWALB_SCALES, "felzenszwalb_cell_px": FELZENSZWALB_CELL_PX,
    "watershed_cells_px": WATERSHED_CELLS_PX, "votes": VOTES, "alphas": ALPHAS, "tune_set": TUNE_SET,
    "miou_guard": MIOU_GUARD, "chosen_method": asdict(chosen[0]) if chosen else None,
    "chosen_vote": asdict(chosen[1]) if chosen else None, "boundary_px": BOUNDARY_IOU_DILATION_PX, "model": "model-lraspp-bn-satellite.pt",
    "latency_tile": [LATENCY_SET, *LATENCY_TOP_LEFT_PX], "latency_threads": LATENCY_THREADS,
}

In [ ]:
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_zoom, fig_grid],
              seed=SEED, device=DEVICE) if LOG_RUN else {"run": "dry"}
grid.to_csv(artifacts / f"grid-{run['run']}.csv", index=False)
grid_latency.to_csv(artifacts / f"latency-{run['run']}.csv", index=False)
summary.to_csv(artifacts / f"summary-{run['run']}.csv")
run["run"]